In [1]:
import numpy as np
import pandas as pd
import synergy_dataset as sd

# Path to your SQLite3 database
db_path = ""
data_path = "./data/"
single_dataset = "van_de_Schoot_2018"

In [2]:
studies = pd.read_json("synergy_studies_validation.jsonl", lines=True)
studies_filtered = studies.sort_values("dataset_id").reset_index(drop=True)
studies_filtered = studies_filtered[studies_filtered["dataset_id"] == single_dataset]
report_order = studies_filtered["dataset_id"].unique()
studies_filtered.reset_index(drop=True, inplace=True)

recall_files = [
    "recalls_d2v_svm.csv",
    "recalls_mxbai_svm.csv",
]
recall_types = ["d2v", "mxbai"]

In [3]:
def get_total_relevant(dataset_id):
    if dataset_id in {"Moran_2021_corrected", "Muthu_2021_corrected"}:
        return pd.read_csv(f"../src/datasets/{dataset_id}_shuffled_raw.csv")[
            "label_included"
        ].sum()
    else:
        return sd.Dataset(dataset_id).to_frame()["label_included"].sum()


# Build the dictionary
total_relevant_dict = {
    dataset_id: get_total_relevant(dataset_id)
    for dataset_id in studies_filtered["dataset_id"].unique()
}

In [4]:
recall_dfs = [pd.read_csv(data_path + f) for f in recall_files]

# Add metadata to each DataFrame
for i, df in enumerate(recall_dfs):
    df["dataset_name"] = studies_filtered["dataset_id"].values
    df["Optimization"] = recall_types[i]
    df["prior_inclusions"] = studies_filtered["prior_inclusions"].apply(len)
    df["prior_exclusions"] = studies_filtered["prior_exclusions"].apply(len)
    df["simulation_id"] = df.groupby("dataset_name").cumcount() + 1

# Combine recall data
df_all = pd.concat(recall_dfs, ignore_index=True)

# Melt dataframe
df_all_melted = df_all.melt(
    id_vars=[
        "dataset_name",
        "Optimization",
        "prior_inclusions",
        "prior_exclusions",
        "simulation_id",
    ],
    var_name="step",
    value_name="recall",
).dropna()

# Convert step to numeric
df_all_melted["step"] = df_all_melted["step"].astype(int)

# Calculate total relevant items
df_all_melted["total_relevant"] = df_all_melted["dataset_name"].map(total_relevant_dict)

# Recall normalization
df_all_melted["relative_recall"] = df_all_melted["recall"] / (
    df_all_melted["total_relevant"] - df_all_melted["prior_inclusions"]
)

# Normalize step values to a common scale [0,1]
df_all_melted["relative_step"] = df_all_melted.groupby(
    ["dataset_name", "Optimization"]
)["step"].transform(lambda x: x / x.max())

In [5]:
df_ah = df_all_melted[df_all_melted["dataset_name"] == single_dataset]
df_ah = df_ah[df_ah["Optimization"] == "d2v"]

df_ah['target_recall'] = df_ah['total_relevant'] - df_ah['prior_inclusions'] -1
df_filtered = df_ah[df_ah['recall'] == df_ah['target_recall']]
lowest_step_df = df_filtered.sort_values(['simulation_id', 'step']).groupby('simulation_id').first().reset_index()

res = lowest_step_df[['simulation_id', 'step', 'recall', 'total_relevant', 'prior_inclusions']]
print(np.mean(res["step"].to_list()))
print(np.std(res["step"].to_list()))

585.8
85.19014027456464


In [6]:
df_ah = df_all_melted[df_all_melted["dataset_name"] == single_dataset]
df_ah = df_ah[df_ah["Optimization"] == "mxbai"]

df_ah['target_recall'] = df_ah['total_relevant'] - df_ah['prior_inclusions']
df_filtered = df_ah[df_ah['recall'] == df_ah['target_recall']]
lowest_step_df = df_filtered.sort_values(['simulation_id', 'step']).groupby('simulation_id').first().reset_index()

res = lowest_step_df[['simulation_id', 'step', 'recall', 'total_relevant', 'prior_inclusions']]
print(np.std(res["step"].to_list()))
print(np.mean(res["step"].to_list()))

45.59045952828289
292.1
